# CareLoop Clinics – No-Show Guard
## Part 3 — Machine Learning: predict no-shows and choose the model honestly

**Input:** `appointments_clean.csv` (Part 1) · **Output:** `test_risk_scores.csv`

Prediction point: the evening before the appointment, after reminders were sent or skipped — so every feature used is known at that moment.
**Rule:** the test set is used for reporting only — never to choose a model, a setting or a threshold.

In [ ]:
# --- Setup: works in Google Colab (uses Google Drive) and locally ---
import os

try:
    from google.colab import drive          # only exists inside Colab
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CareLoop-NoShow-Guard"
    os.makedirs(os.path.join(PROJECT_DIR, "data"), exist_ok=True)
    os.chdir(PROJECT_DIR)                    # all relative paths start here
except ImportError:
    pass                                     # running locally: use the current folder

print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)
import sklearn
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| NumPy", np.__version__)

df = pd.read_csv("appointments_clean.csv")   # NOT shuffled or re-sorted: row order fixes the split
print(df.shape)

### Task 2.2 cross-check (Python vs SQL)
Clinic appointment counts and no-show counts with pandas `groupby`, to confirm they match the SQL result in `02_queries.sql`.

In [ ]:
clinics = pd.read_csv("clinics_enriched.csv")
by_clinic = (df.groupby("clinic_id")
               .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
               .reset_index()
               .merge(clinics[["clinic_id", "clinic_name", "city"]], on="clinic_id"))
by_clinic["no_show_pct"] = (100 * by_clinic["no_shows"] / by_clinic["appointments"]).round(1)
by_clinic.sort_values("no_show_pct", ascending=False)[["clinic_name", "city", "appointments", "no_shows", "no_show_pct"]]

## Task 3.1 — Features, split and baseline

In [ ]:
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print("Feature columns:", X.shape[1])
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("\nTraining rows:", len(X_train), "| Test rows:", len(X_test))
print("No-shows in test set:", y_test.sum())

baseline_acc = (y_test == 0).mean()    # always predict "Attended"
print(f"Baseline accuracy (always 'Attended'): {baseline_acc:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

## Task 3.2 — Logistic regression
The scaler is fitted on the **training set only**; the test set is only transformed with it.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

lr_pred = log_reg.predict(X_test_s)
lr_prob = log_reg.predict_proba(X_test_s)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, lr_pred):.3f}")
print(f"Precision: {precision_score(y_test, lr_pred):.3f}")
print(f"Recall   : {recall_score(y_test, lr_pred):.3f}")
print(f"F1       : {f1_score(y_test, lr_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, lr_prob):.3f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, lr_pred))

In [ ]:
coefs = pd.DataFrame({"feature": X.columns, "coefficient": log_reg.coef_[0]}).round(3)
print(coefs.sort_values("coefficient").to_string(index=False))

i = list(X.columns).index("reminder_sent")
coef_reminder = log_reg.coef_[0][i]
sd_reminder = scaler.scale_[i]
odds_per_sd = np.exp(coef_reminder)
odds_no_to_yes = np.exp(coef_reminder / sd_reminder)
print(f"\nreminder_sent coefficient (scaled): {coef_reminder:.3f}, SD of reminder_sent: {sd_reminder:.3f}")
print(f"np.exp(coef)          = {odds_per_sd:.2f}  (odds multiplier per 1 standard deviation)")
print(f"np.exp(coef / scale_) = {odds_no_to_yes:.2f}  (odds multiplier for No -> Yes)")
print(f"-> about {(1 - odds_no_to_yes) * 100:.0f}% lower odds of a no-show when a reminder was sent")

**Interpretation.** The two numbers differ because the model was fitted on *standardised* features: `np.exp(coef)` is the odds change for a one-standard-deviation step (≈ 0.49 on a 0/1 variable), whereas dividing by `scaler.scale_` converts it back to the real step from No (0) to Yes (1), which is about twice as large.

In this model, **holding the other features constant**, appointments with a reminder have roughly 69% lower odds of a no-show (odds × ≈0.31). This is an **association, not proof of cause**: the data are observational, so reminders may simply have gone to patients who were already more likely to attend.

## Task 3.3 — Decision tree and overfitting

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_d3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, model in [("Full-depth tree", tree_full), ("Depth-3 tree", tree_d3)]:
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: depth={model.get_depth()}, train accuracy={tr:.3f}, test accuracy={te:.3f}, gap={tr - te:.3f}")
print(f"Baseline accuracy: {baseline_acc:.4f}")

**What the gap shows:** the unlimited tree scores 100% on training data but falls *below the 0.7525 baseline* on the test set — it has memorised noise in the training rows (overfitting), while the depth-3 tree keeps a small train/test gap and generalises better.

## Task 3.4 — Random forest and a comparison table

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print(f"Random forest: train accuracy={accuracy_score(y_train, rf.predict(X_train)):.3f}, "
      f"test accuracy={accuracy_score(y_test, rf_pred):.3f}, recall={recall_score(y_test, rf_pred):.3f}, "
      f"ROC-AUC={roc_auc_score(y_test, rf_prob):.3f}")

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop three feature importances:")
print(importances.head(3).round(3))

In [ ]:
tree_prob = tree_d3.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, tree_d3.predict(X_test)), accuracy_score(y_test, rf_pred)],
    "recall": [recall_score(y_test, lr_pred), recall_score(y_test, tree_d3.predict(X_test)), recall_score(y_test, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, lr_prob), roc_auc_score(y_test, tree_prob), roc_auc_score(y_test, rf_prob)],
}).round(3)
comparison

Logistic regression looks best in this table: it has the highest ROC-AUC (it ranks risky bookings best) and the best accuracy, while the random forest overfits (train accuracy 1.00). This table is for reporting only — the formal choice is made in Task 3.5 using training data only.

## Task 3.5 — Tune, choose the model and the threshold, then test once
### (a) Grid search on the training set

In [ ]:
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print(f"best_score_ (cross-validated ROC-AUC): {grid.best_score_:.3f}")

### (b) Choose the model with training cross-validation only
Logistic regression uses the supplied pipeline so the scaler is re-fitted inside every fold (no leakage from the validation fold).

In [ ]:
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

cv_results = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100 trees)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest": grid.best_score_,
}
cv_table = pd.Series(cv_results, name="cv_roc_auc").round(3).sort_values(ascending=False)
print(cv_table)
print("\nWinner:", cv_table.index[0])

### (c) Choose the threshold with out-of-fold training probabilities

In [ ]:
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, flag),
                 "recall": recall_score(y_train, flag),
                 "f1": f1_score(y_train, flag),
                 "flagged": int(flag.sum())})
threshold_table = pd.DataFrame(rows).round(3)
print(threshold_table.to_string(index=False))

THRESHOLD = threshold_table.loc[threshold_table["f1"].idxmax(), "threshold"]
print("\nChosen threshold (higher F1):", THRESHOLD)

**Why recall matters here:** a missed warning (false negative) costs an empty doctor slot, while an extra phone call (false positive) costs the front desk about a minute — so catching more real no-shows is worth some extra calls. 0.35 has the higher F1 and much higher recall, so it is chosen.

### (d) Fit the winner on the training set and evaluate **once** on the test set

In [ ]:
lr_pipe.fit(X_train, y_train)                 # the pipeline fits its own scaler on training data
probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flag = (probs >= THRESHOLD).astype(int)

print(f"Final model: logistic regression, threshold {THRESHOLD}")
print(f"Precision: {precision_score(y_test, test_flag):.3f}")
print(f"Recall   : {recall_score(y_test, test_flag):.3f}")
print(f"Flagged  : {test_flag.sum()} of {len(y_test)} test appointments")
print(f"ROC-AUC  : {roc_auc_score(y_test, probs):.3f}")
print(f"(For information only: tuned forest test ROC-AUC = {roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]):.3f})")

### (e) Save the test-set risk scores

In [ ]:
risk = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs.round(4),
    "flag": test_flag,
})
risk.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv:", risk.shape, "| flagged rows:", risk["flag"].sum())
risk.sort_values("risk_prob", ascending=False).head()

**How a clinic would really choose the threshold:** by how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slot — e.g. lower the threshold if staff have spare call capacity, raise it if they can only call a handful of patients.